In [1]:
import numpy as np
import pandas as pd
import lightgbm as lgb
from sklearn.metrics import r2_score
from sklearn.model_selection import KFold
import warnings
warnings.filterwarnings('ignore')

TRAIN_PATH = "/Users/pashantraj/Desktop/flipkart/dataset/train.csv"
TEST_PATH  = "/Users/pashantraj/Desktop/flipkart/dataset/test.csv"

In [2]:
_BASE32  = '0123456789bcdefghjkmnpqrstuvwxyz'
_DEC_MAP = {c: i for i, c in enumerate(_BASE32)}

def geohash_decode(gh):
    bits = []
    for c in gh:
        v = _DEC_MAP[c]
        for i in range(4, -1, -1):
            bits.append((v >> i) & 1)
    la, lb, loa, lob = -90., 90., -180., 180.
    for i, b in enumerate(bits):
        if i % 2 == 0:
            m = (loa + lob) / 2
            if b: loa = m
            else: lob = m
        else:
            m = (la + lb) / 2
            if b: la = m
            else: lb = m
    return (la + lb) / 2, (loa + lob) / 2

def geohash_encode(lat, lon, precision=6):
    """Re-encode a lat/lon pair back to a geohash of given precision."""
    base32 = '0123456789bcdefghjkmnpqrstuvwxyz'
    la, lb, loa, lob = -90., 90., -180., 180.
    bits, bit, ch, result = 0, 0, 0, ''
    even = True
    while len(result) < precision:
        if even:
            m = (loa + lob) / 2
            if lon >= m: ch = (ch << 1) | 1; loa = m
            else: ch = ch << 1; lob = m
        else:
            m = (la + lb) / 2
            if lat >= m: ch = (ch << 1) | 1; la = m
            else: ch = ch << 1; lb = m
        even = not even
        bits += 1
        if bits == 5:
            result += base32[ch]
            bits = 0; ch = 0
    return result

def build_neighbor_map(all_geohashes, loc_demand_map):
    """
    For each geohash, find up to 4 geohash neighbors by shifting lat/lon.
    """
    DELTA = 0.0055
    decoded = {gh: geohash_decode(gh) for gh in all_geohashes}
    gh_set = set(all_geohashes)
    neighbor_means = {}
    for gh, (lat, lon) in decoded.items():
        neighbors = []
        for dlat, dlon in [(DELTA,0),(-DELTA,0),(0,DELTA),(0,-DELTA)]:
            candidate = geohash_encode(lat+dlat, lon+dlon, precision=6)
            if candidate in gh_set and candidate != gh:
                neighbors.append(candidate)
        if neighbors:
            vals = [loc_demand_map.get(n, np.nan) for n in neighbors]
            vals = [v for v in vals if not np.isnan(v)]
            neighbor_means[gh] = np.mean(vals) if vals else np.nan
        else:
            neighbor_means[gh] = np.nan
    return neighbor_means

In [3]:
print("Loading data...")

df_train = pd.read_csv(TRAIN_PATH)
df_test  = pd.read_csv(TEST_PATH)

df_train['is_train'] = 1
df_test['is_train']  = 0
df_test['demand']    = np.nan

df_full = pd.concat([df_train, df_test], ignore_index=True)

Loading data...


In [4]:
print("Engineering features...")

# A. Cyclical Time + structured bins
df_full['hour']   = df_full['timestamp'].str.split(':').str[0].astype(int)
df_full['minute'] = df_full['timestamp'].str.split(':').str[1].astype(int)
mins_of_day       = df_full['hour'] * 60 + df_full['minute']

df_full['time_sin'] = np.sin(2 * np.pi * mins_of_day / 1440)
df_full['time_cos'] = np.cos(2 * np.pi * mins_of_day / 1440)
df_full['time_bucket'] = mins_of_day // 15

df_full['is_rush_hour'] = (
    ((df_full['hour'] >= 7) & (df_full['hour'] < 9)) |
    ((df_full['hour'] >= 17) & (df_full['hour'] < 19))
).astype(int)

df_full['is_peak_window'] = (
    (df_full['hour'] >= 6) & (df_full['hour'] < 20)
).astype(int)

# B. Geohash Decoded Lat/Lon
coords = df_full['geohash'].map(geohash_decode)
df_full['lat'] = coords.map(lambda x: x[0])
df_full['lon'] = coords.map(lambda x: x[1])

# C. Golden feature: demand from yesterday (Safe Lag)
lookup_dict = df_train.set_index(['geohash', 'timestamp', 'day'])['demand'].to_dict()

df_full['demand_yesterday'] = df_full.apply(
    lambda row: lookup_dict.get((row['geohash'], row['timestamp'], row['day'] - 1), np.nan),
    axis=1
)

# D. Location & timestamp aggregate stats (from train only)
loc_stats = df_train.groupby('geohash')['demand'].agg(
    loc_mean='mean', loc_std='std', loc_median='median',
    loc_max='max', loc_p25=lambda x: x.quantile(0.25),
    loc_p75=lambda x: x.quantile(0.75)
).reset_index()

ts_stats = df_train.groupby('timestamp')['demand'].agg(
    ts_mean='mean', ts_std='std', ts_median='median',
    ts_max='max'
).reset_index()

df_full = df_full.merge(loc_stats, on='geohash', how='left')
df_full = df_full.merge(ts_stats, on='timestamp', how='left')

stat_cols = ['loc_mean', 'loc_std', 'loc_median', 'loc_max', 'loc_p25', 'loc_p75',
             'ts_mean', 'ts_std', 'ts_median', 'ts_max']
df_full[stat_cols] = df_full[stat_cols].fillna(0)

# Fallback for yesterday's demand
df_full['demand_yesterday'] = df_full['demand_yesterday'].fillna(df_full['loc_mean'])

# E. Neighbor demand (spatial spillover)
print("Computing geohash neighbor features...")
all_geohashes = df_full['geohash'].unique()
loc_demand_map = df_train.groupby('geohash')['demand'].mean().to_dict()
neighbor_map = build_neighbor_map(all_geohashes, loc_demand_map)
df_full['neighbor_mean_demand'] = df_full['geohash'].map(neighbor_map)
df_full['neighbor_mean_demand'] = df_full['neighbor_mean_demand'].fillna(df_full['loc_mean'])

# F. Safe Interaction / ratio features
eps = 1e-6
df_full['momentum_ratio']     = df_full['demand_yesterday'] / (df_full['loc_mean'] + eps)
df_full['ts_vs_loc']          = df_full['ts_mean']          / (df_full['loc_mean'] + eps)
df_full['neighbor_vs_loc']    = df_full['neighbor_mean_demand'] / (df_full['loc_mean'] + eps)
df_full['iqr_range']          = df_full['loc_p75'] - df_full['loc_p25']

# G. Categorical features
df_full['RoadType']      = df_full['RoadType'].fillna('Unknown').astype('category')
df_full['Weather']       = df_full['Weather'].fillna('Unknown').astype('category')
df_full['LargeVehicles'] = df_full['LargeVehicles'].fillna('Unknown').astype('category')
df_full['Landmarks']     = df_full['Landmarks'].fillna('Unknown').astype('category')

df_full['NumberofLanes'] = df_full['NumberofLanes'].fillna(df_full['NumberofLanes'].median())
df_full['Temperature']   = df_full['Temperature'].fillna(df_full['Temperature'].median())

# H. Temperature bucketing
df_full['temp_bin'] = pd.cut(df_full['Temperature'],
                              bins=[-np.inf, 5, 15, 25, 35, np.inf],
                              labels=[0, 1, 2, 3, 4]).astype(int)

Engineering features...
Computing geohash neighbor features...


In [5]:
train_df = df_full[(df_full['is_train'] == 1) & (df_full['demand'].notnull())].copy()
test_df  = df_full[df_full['is_train'] == 0].copy()

# Notice loc_ts_mean and derived ratios are GONE.
FEATURES = [
    'time_sin', 'time_cos', 'time_bucket', 'is_rush_hour', 'is_peak_window',
    'lat', 'lon',
    'loc_mean', 'loc_std', 'loc_median', 'loc_max', 'loc_p25', 'loc_p75', 'iqr_range',
    'ts_mean', 'ts_std', 'ts_median', 'ts_max',
    'neighbor_mean_demand',
    'demand_yesterday',
    'momentum_ratio', 'ts_vs_loc', 'neighbor_vs_loc',
    'RoadType', 'NumberofLanes', 'LargeVehicles', 'Landmarks',
    'Weather', 'Temperature', 'temp_bin',
]

X      = train_df[FEATURES]
y      = np.log1p(train_df['demand'])  
X_test = test_df[FEATURES]

print(f"\nTraining on {len(X)} rows with {len(FEATURES)} features. Predicting {len(X_test)} rows.")


Training on 77299 rows with 30 features. Predicting 41778 rows.


In [6]:
print("\nStarting 5-Fold LightGBM Training with OOF target encoding...")

params = {
    'objective':       'regression',
    'metric':          'rmse',
    'learning_rate':   0.015,
    'max_depth':       7,
    'num_leaves':      48,          
    'feature_fraction': 0.65,       
    'bagging_fraction': 0.80,
    'bagging_freq':    3,
    'min_data_in_leaf': 50,         
    'lambda_l1':       1.0,         
    'lambda_l2':       2.0,         
    'verbose':        -1,
    'seed':            42
}

kf = KFold(n_splits=5, shuffle=True, random_state=42)

oof_preds  = np.zeros(len(train_df))
test_preds = np.zeros(len(test_df))

train_df = train_df.reset_index(drop=True)
X        = train_df[FEATURES].copy()

SMOOTH = 10  

for fold, (train_idx, val_idx) in enumerate(kf.split(X, y)):
    X_tr_fold = X.iloc[train_idx].copy()
    y_tr_fold = y.iloc[train_idx]
    X_va_fold = X.iloc[val_idx].copy()
    X_te_fold = X_test.copy()

    # ── Safe OOF Geohash Target Encoding ─────────────────────
    gh_col_train = train_df['geohash'].iloc[train_idx]
    enc_df = pd.DataFrame({'geohash': gh_col_train, 'target': y_tr_fold.values})
    global_mean = y_tr_fold.mean()

    gh_stats = enc_df.groupby('geohash')['target'].agg(['mean', 'count'])
    gh_stats['encoded'] = (
        (gh_stats['mean'] * gh_stats['count'] + global_mean * SMOOTH)
        / (gh_stats['count'] + SMOOTH)
    )
    gh_enc_map = gh_stats['encoded'].to_dict()

    gh_val  = train_df['geohash'].iloc[val_idx]
    gh_test = test_df['geohash']

    X_tr_fold['geohash_target_enc'] = gh_col_train.map(gh_enc_map).fillna(global_mean).values
    X_va_fold['geohash_target_enc'] = gh_val.map(gh_enc_map).fillna(global_mean).values
    X_te_fold['geohash_target_enc'] = gh_test.map(gh_enc_map).fillna(global_mean).values
    # ─────────────────────────────────────────────────────────

    feat_cols = FEATURES + ['geohash_target_enc']

    train_ds = lgb.Dataset(X_tr_fold[feat_cols], label=y_tr_fold)
    val_ds   = lgb.Dataset(X_va_fold[feat_cols], label=y.iloc[val_idx], reference=train_ds)

    model = lgb.train(
        params,
        train_ds,
        num_boost_round=4000,
        valid_sets=[train_ds, val_ds],
        callbacks=[
            lgb.early_stopping(stopping_rounds=100, verbose=False),
            lgb.log_evaluation(period=0)
        ]
    )

    # Expm1 to reverse the target transformation
    fold_preds = np.expm1(model.predict(X_va_fold[feat_cols]))
    oof_preds[val_idx] = fold_preds

    test_preds += np.expm1(model.predict(X_te_fold[feat_cols])) / kf.n_splits

    fold_r2 = r2_score(np.expm1(y.iloc[val_idx]), fold_preds)
    print(f"Fold {fold+1} R2: {fold_r2:.4f} (Trees: {model.best_iteration})")

overall_r2 = r2_score(np.expm1(y), oof_preds)
print(f"\n=> 5-Fold CV Overall R2: {overall_r2:.4f}")

# Feature importance 
feat_cols_final = FEATURES + ['geohash_target_enc']
fi = pd.DataFrame({
    'Feature': feat_cols_final,
    'Gain':    model.feature_importance(importance_type='gain')
}).sort_values('Gain', ascending=False)
print("\nTop 15 Features by Gain:")
print(fi.head(15).to_string(index=False))




Starting 5-Fold LightGBM Training with OOF target encoding...
Fold 1 R2: 0.9548 (Trees: 3999)
Fold 2 R2: 0.9563 (Trees: 3999)
Fold 3 R2: 0.9562 (Trees: 3999)
Fold 4 R2: 0.9519 (Trees: 4000)
Fold 5 R2: 0.9548 (Trees: 4000)

=> 5-Fold CV Overall R2: 0.9548

Top 15 Features by Gain:
           Feature         Gain
          RoadType 10540.913941
geohash_target_enc  2366.989531
          loc_mean  1975.884126
        loc_median   565.263025
           loc_p75   380.049151
  demand_yesterday   357.826982
         ts_median   340.500514
           ts_mean   312.097608
            ts_std   300.241482
     LargeVehicles   231.597249
           loc_p25   202.220091
           loc_max   148.203617
          time_sin    96.693332
    momentum_ratio    88.650400
     NumberofLanes    87.787532


In [7]:
print("\nPreparing submission...")

test_preds = np.maximum(test_preds, 0)

submission = pd.read_csv(TEST_PATH)[['Index']].copy()
pred_map   = dict(zip(test_df['Index'].values, test_preds))
submission['demand'] = submission['Index'].map(pred_map)

missing = submission['demand'].isna().sum()
if missing > 0:
    print(f"Warning: Filling {missing} missing rows with global train mean.")
    submission['demand'] = submission['demand'].fillna(np.expm1(y).mean())

submission.to_csv("submission_final.csv", index=False)
print(f"Saved submission_final.csv ({len(submission)} rows)")


Preparing submission...
Saved submission_final.csv (41778 rows)
